# NAVDRIFT-0 | ISRO SIH 2026 PS #26168
## Master Training Notebook

**Upload all `navdrift_0X_*.py` files to `/content/` before running.**

### Estimated A100 time budget:
| Stage | Script | Hours |
|-------|--------|-------|
| Setup + Data | 00, 01 | 6 |
| DRIFTFormer | 02 | 18 |
| IMU Denoiser | 03 | 12 |
| Adaptive EKF | 04 | 6 |
| Tunnel Detector | 05 | 4 |
| NavIC DOP | 06 | 2 |
| ONNX Export | 07 | 2 |
| Validation | 08 | 2 |
| **Total** | | **~52 hrs** |

Each cell saves to Google Drive. **Safe to disconnect and resume anytime.**

---
### Resume after disconnect:
Just re-run from the cell where you left off. Checkpoints auto-load from Drive.

In [ ]:
# ============================================================
# CELL 0 — UPLOAD CHECK
# Run this first to verify all scripts are uploaded.
# ============================================================
import os
scripts = [
    'navdrift_00_setup.py',
    'navdrift_01_data_pipeline.py',
    'navdrift_02_driftformer.py',
    'navdrift_03_imu_denoiser.py',
    'navdrift_04_adaptive_ekf.py',
    'navdrift_05_tunnel_detector.py',
    'navdrift_06_navic_dop.py',
    'navdrift_07_onnx_export.py',
    'navdrift_08_validate.py',
]
for s in scripts:
    exists = os.path.exists(f'/content/{s}')
    print(f"{'OK' if exists else 'MISSING'}  {s}")

In [ ]:
# ============================================================
# CELL 1 — SETUP
# Installs deps, mounts Drive, starts keepalive.
# ~5 minutes. Run once per session.
# ============================================================
%run /content/navdrift_00_setup.py

In [ ]:
# ============================================================
# CELL 2 — DATA PIPELINE
# Downloads EuRoC, TUM-VI, NCLT, NavIC RINEX.
# Builds unified HDF5 on Drive.
# ~4-6 hours (download + processing).
#
# KAIST: see instructions printed below — needs free account.
# Upload KAIST zips to Drive first, then re-run this cell.
# ============================================================
%run /content/navdrift_01_data_pipeline.py

In [ ]:
# ============================================================
# CELL 3 — DRIFTFORMER
# 4L-8H causal transformer. 529K params.
# ~18 hours on A100 for 200 epochs.
# SAFE TO STOP: checkpoints every epoch to Drive.
# Resume: just rerun this cell — auto-loads latest checkpoint.
# ============================================================
%run /content/navdrift_02_driftformer.py

In [ ]:
# ============================================================
# CELL 4 — IMU DENOISER (TCN)
# ~12 hours on A100 for 150 epochs.
# Resume: rerun this cell.
# ============================================================
%run /content/navdrift_03_imu_denoiser.py

In [ ]:
# ============================================================
# CELL 5 — ADAPTIVE EKF PREDICTOR
# ~6 hours on A100 for 300 epochs.
# Resume: rerun this cell.
# ============================================================
%run /content/navdrift_04_adaptive_ekf.py

In [ ]:
# ============================================================
# CELL 6 — TUNNEL DETECTOR (Bi-LSTM)
# ~4 hours on A100 for 200 epochs.
# Resume: rerun this cell.
# ============================================================
%run /content/navdrift_05_tunnel_detector.py

In [ ]:
# ============================================================
# CELL 7 — NAVIC DOP PREDICTOR
# ~2 hours on A100 for 500 epochs.
# Resume: rerun this cell.
# ============================================================
%run /content/navdrift_06_navic_dop.py

In [ ]:
# ============================================================
# CELL 8 — ONNX INT8 EXPORT + BENCHMARK
# Exports all 5 models. Benchmarks combined inference.
# Target: <8ms total pipeline on CPU (ARM Cortex-A72 proxy).
# ~30 minutes.
# ============================================================
%run /content/navdrift_07_onnx_export.py

In [ ]:
# ============================================================
# CELL 9 — END-TO-END VALIDATION
# Runs full pipeline on test sequences.
# Generates compliance_curve.png + isro_benchmark_table.csv.
# ISRO target: drift <10% for >90% of steps.
# ~1-2 hours.
# ============================================================
%run /content/navdrift_08_validate.py

In [ ]:
# ============================================================
# CELL 10 — SHOW RESULTS
# ============================================================
import json
from pathlib import Path
from IPython.display import Image, display

results_dir = Path('/content/drive/MyDrive/NAVDRIFT0/results')

# Compliance metrics
with open(results_dir / 'validation_full.json') as f:
    val = json.load(f)

m = val['compliance_metrics']
print('=' * 50)
print('  NAVDRIFT-0 ISRO COMPLIANCE RESULTS')
print('=' * 50)
print(f"  ATE RMSE:                {m['ATE_RMSE_m']} m")
print(f"  Mean Drift:              {m['drift_mean_pct']} %")
print(f"  Max Drift:               {m['drift_max_pct']} %")
print(f"  Steps under 10% target:  {m['pct_under_10_target']} %")
print(f"  Steps under 5%:          {m['pct_under_5']} %")
print(f"  ISRO PASS:               {m['isro_pass']}")

# Show plot
display(Image(str(results_dir / 'compliance_curve.png')))

In [ ]:
# ============================================================
# CELL 11 — DOWNLOAD MODELS TO LOCAL
# Run after training to download models to your machine.
# ============================================================
from google.colab import files
from pathlib import Path

models_dir = Path('/content/drive/MyDrive/NAVDRIFT0/models')
to_download = [
    'driftformer_int8.onnx',
    'imu_denoiser_int8.onnx',
    'adaptive_ekf_int8.onnx',
    'tunnel_det_int8.onnx',
    'navic_dop_int8.onnx',
    'models_manifest.json',
]
for name in to_download:
    p = models_dir / name
    if p.exists():
        files.download(str(p))
        print(f'Downloaded: {name}')
    else:
        print(f'Missing:    {name}')